
 GOLD LAYER — DAILY PRODUCT SALES
 Table: gold.daily_product_sales

 Grain:
 1 row = 1 product per day

 Business goal:
 Analyze product performance over time:
 - Daily product revenue
 - Product sales trends
 - Product seasonality


1. Load Source Data

In [0]:


df_fact = spark.table("gold.fact_sales")

2. Daily Product Aggregation

In [0]:


from pyspark.sql.functions import sum, count

df_daily_product = df_fact.groupBy(
    "order_date",
    "product_id",
    "product_name",
    "category"
).agg(
    sum("revenue").alias("daily_revenue"),
    sum("quantity").alias("daily_quantity"),
    count("order_id").alias("total_orders")
)
df_daily_product = df_daily_product.drop("date")

3. Join with Date Dimension

In [0]:

df_date = spark.table("gold.dim_date")

df_daily_product = df_daily_product.join(
    df_date,
    df_daily_product["order_date"] == df_date["date"],
    "left"
)

4. Clean Columns

In [0]:

# Remove duplicate date column from dim_date
df_daily_product = df_daily_product.drop("date")

5. Data Quality Checks

In [0]:

null_products = df_daily_product.filter("product_id IS NULL").count()
negative_revenue = df_daily_product.filter("daily_revenue < 0").count()

print(f"Null product_id: {null_products}")
print(f"Negative revenue rows: {negative_revenue}")

6. Write to Gold Table

In [0]:

spark.sql("DROP TABLE IF EXISTS gold.daily_product_sales")

df_daily_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.daily_product_sales")

7. Validation

In [0]:


display(
    spark.table("gold.daily_product_sales")
    .orderBy("order_date", "product_id")
)